# Reflection as a Cyclic Graph

**What you will build:** the reflection loop from 0.5 (generate -> critique -> improve) as a LangGraph **cycle**. This is where a graph beats both plain Python and n8n: the loop is a first-class, drawable structure with a clean stop condition. Maps to chapter 05 of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/25_reflection_cyclic_graph.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## Generate, critique, loop

Two nodes and a conditional edge that can point *back*: `generate` writes a draft (using any feedback so far), `critique` judges it, and the router either loops back to `generate` or ends. We cap iterations so it always terminates.

The critic is **hybrid**, exactly per 0.5's hierarchy: the objective constraints (exactly 3 sentences, under 25 words) are *Python* — free, instant, and the feedback can quote exact counts — and only when structure passes does an LLM judge the subjective part (is it punchy?). Moving the loop into a graph changed the *plumbing*, never the rule about who checks what.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    task: str
    draft: str
    feedback: str
    passed: bool
    iterations: int

def generate(state: State) -> dict:
    fb = state.get("feedback", "")
    prompt = f"Task: {state['task']}"
    if fb:
        prompt += f"\nRevise your previous attempt using this feedback: {fb}\nPrevious: {state['draft']}"
    r = llm.invoke([{"role": "system", "content": "You are a concise copywriter."},
                    {"role": "user", "content": prompt}])
    return {"draft": r.content, "iterations": state.get("iterations", 0) + 1}

import re

def critique(state: State) -> dict:
    draft = state["draft"].strip()
    # 0.5's hierarchy, unchanged by the graph: objective rules are CODE (free, exact),
    # the LLM judges only the subjective residue.
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", draft) if s]
    words = len(draft.split())
    if len(sentences) != 3 or words >= 25:
        return {"passed": False,
                "feedback": (f"Structure check failed: {len(sentences)} sentences, {words} words. "
                             "Rewrite as EXACTLY 3 sentences, under 25 words total.")}
    r = llm.invoke([{"role": "system", "content":
                     "You are a strict editor judging PUNCH, not format (the format is already verified). "
                     "Reply exactly 'PASS' if the blurb is vivid and compelling; otherwise one sentence "
                     "of specific feedback."},
                    {"role": "user", "content": draft}])
    verdict = r.content.strip()
    return {"passed": verdict.upper().startswith("PASS"), "feedback": verdict}

def route(state: State) -> str:
    if state["passed"] or state["iterations"] >= 3:   # stop condition
        return END
    return "generate"                                   # loop back to improve

In [ ]:
builder = StateGraph(State)
builder.add_node("generate", generate)
builder.add_node("critique", critique)
builder.add_edge(START, "generate")
builder.add_edge("generate", "critique")
builder.add_conditional_edges("critique", route, {"generate": "generate", END: END})

graph = builder.compile()

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())

The drawn graph has a visible cycle `critique -> generate`. And since the whole point of this chapter is a loop, don't run it blind — **stream it** (2.2) and watch every iteration as it happens:

In [ ]:
final = {}
for chunk in graph.stream(
    {"task": "Write a product blurb for a noise-cancelling coffee mug."},
    stream_mode="updates",
):
    for node, update in chunk.items():
        final.update(update)
        if node == "generate":
            print(f"[generate | attempt {update['iterations']}] {update['draft'][:70]!r}")
        else:
            status = "PASS" if update["passed"] else "needs work"
            print(f"[critique | {status:10s}] {update['feedback'][:70]}")

print("\niterations:", final["iterations"])
print("final:     ", final["draft"])

Read the stream: attempt, verdict, attempt, verdict — the cycle from the drawing, executing. Notice *which* check fails on early attempts (usually the structure one, with exact counts in the feedback) versus later ones (the judge). Same pattern as 0.5, but now the loop *is* the graph — inspectable, drawable, with the stop condition in one obvious place. Cycles are native to LangGraph: in n8n the loop-back needs an If node and careful wiring, whereas here it's a single edge — the reason a stateful graph earns its keep for this kind of flow.

```{note}
As in 0.5, a code check (`iterations >= 3`) guarantees termination, while the critic handles quality. Always give a cyclic graph a hard cap — it's the graph equivalent of `max_turns`.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Measure the non-determinism.** Run the stream cell three times and note the iteration counts. **Done when:** you have three (likely different) numbers and can say why an eval (1.6) would quote the *average*, not one run.
2. **Add a constraint to the code check.** The blurb must also contain the word "quiet". **Done when:** you see a run where the structure passes, the new rule fails, and the feedback names the missing word — code checks compose for free.
3. **Price the loop.** Count LLM calls per run (one per `generate`, plus one per `critique` *that reaches the judge*) and compute the cost multiplier versus a single generation. **Done when:** you can finish the sentence "reflection made this N× more expensive, worth it because…" honestly.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**2 —**

```python
    if len(sentences) != 3 or words >= 25 or "quiet" not in draft.lower():
        return {"passed": False,
                "feedback": (f"Structure check failed: {len(sentences)} sentences, {words} words, "
                             f"contains 'quiet': {'quiet' in draft.lower()}. "
                             "EXACTLY 3 sentences, under 25 words, and include the word 'quiet'.")}
```

One boolean, zero tokens, and the feedback quotes the precise state of every rule — try getting that reliability from a judge prompt.

**3 —** The hybrid critic's economics: a structure failure costs **one** call per iteration (generate only — the judge never runs); only structurally-valid drafts pay for judging. Typical run: 2–3 generates + 1–2 judge calls ≈ 4–5× a single generation. The 0.5 hierarchy isn't just about reliability — putting code first also makes every failed iteration cheaper.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| Never passes, always hits the cap | Constraints genuinely hard, or the reviser drops one rule while fixing another | Include ALL constraints in every revision prompt (the feedback here does); raise the cap consciously |
| Passes on attempt 1 every time | Task too easy to show the loop | Tighten the constraints — this demo *wants* visible failure (0.5's tagline trick) |
| The judge flip-flops (PASS, then feedback on identical text) | Subjective judging is noisy | Keep temperature 0; make the rubric concrete; remember the code check is the only *stable* gate |
| `GraphRecursionError` before your cap | Each generate+critique round is 2 super-steps; default limits can bite first | Count steps like 2.2 taught (2 × iterations + 1) or raise `recursion_limit` |
::::

**What's next:** in **2.6** we go from one agent to **many** — a supervisor that routes work to specialist agents.